# 02 - CLIP/OpenCLIP Preprocessing And Embeddings

Genera embeddings visuales desde crops y embeddings textuales desde nombres naturales de clase calculado en promedio de prompt previmaente definidos.

In [3]:
%pip install numpy pandas Pillow tqdm torch torchvision open_clip_torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
import open_clip

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MODEL_NAME = "ViT-B-32"  # Alternatives: "ViT-B-16", "RN50"
PRETRAINED = "laion2b_s34b_b79k"
BATCH_SIZE = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_PROMPT_NAMES = {
    "0": "gun",
    "1": "knife",
    "2": "pliers",
    "3": "scissors",
    "4": "wrench",
}

PROMPT_TEMPLATES = [
    "a baggage x-ray image of a {}",
    "an x-ray crop containing a {}",
    "a prohibited item: {}",
    "a security x-ray image showing a {}",
    "a cropped x-ray baggage image of a {}",
]

print(f"Device: {DEVICE}")

Device: cpu


In [5]:
crops_metadata_path = OUTPUTS_DIR / "crops_metadata.csv"
all_metadata = pd.read_csv(crops_metadata_path)
all_metadata["class_id"] = all_metadata["class_id"].astype(str)
all_metadata["class_name"] = all_metadata["class_id"].map(CLASS_PROMPT_NAMES)
assert all_metadata["class_name"].notna().all(), "Found class_id without official class_name mapping"

# El banco de embeddings visuales se construye SOLO con train + valid.
# El split test queda reservado para consultas/evaluacion image-to-image.
metadata = all_metadata[all_metadata["split"].isin(["train", "valid"])].reset_index(drop=True)
test_metadata = all_metadata[all_metadata["split"].eq("test")].reset_index(drop=True)

assert not metadata.empty, "No train/valid crops found for the retrieval index"
assert set(metadata["split"].unique()).issubset({"train", "valid"})
print(f"Index crops train+valid: {metadata.shape}")
print(f"Held-out test crops for queries/evaluation: {test_metadata.shape}")
metadata.head()

Index crops train+valid: (7947, 17)
Held-out test crops for queries/evaluation: (883, 17)


,crop_id,split,image_path,label_path,crop_path,class_id,class_name,x1,y1,x2,y2,bbox_width,bbox_height,bbox_area,image_width,image_height,relative_area
0,train_009000_jpg.rf.8c46e1aa5b46a0ad24ee4bcb29...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009000_jpg.rf.8c46e1aa5...,2,pliers,258,210,286,241,28,31,868,416,416,0.005016
1,train_009002_jpg.rf.18bf80f2cfdb51f853da15019f...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009002_jpg.rf.18bf80f2c...,2,pliers,219,125,245,157,26,32,832,416,416,0.004808
2,train_009003_jpg.rf.46963402c4cb6f46a47e508b89...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009003_jpg.rf.46963402c...,2,pliers,153,145,199,171,46,26,1196,416,416,0.006911
3,train_009007_jpg.rf.a5143afbb0c741f3b60fc72403...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009007_jpg.rf.a5143afbb...,2,pliers,154,105,180,187,26,82,2132,416,416,0.012320
4,train_009012_jpg.rf.bc99877ade8754d2be89119361...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009012_jpg.rf.bc99877ad...,2,pliers,322,144,347,178,25,34,850,416,416,0.004912


In [6]:
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED, device=DEVICE)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()
print(f"Loaded {MODEL_NAME} / {PRETRAINED}")

Loaded ViT-B-32 / laion2b_s34b_b79k


In [7]:
def load_preprocessed_crop(relative_crop_path):
    crop_path = Path(relative_crop_path)
    if not crop_path.is_absolute():
        crop_path = PROJECT_ROOT / crop_path
    with Image.open(crop_path) as image:
        image = image.convert("RGB")
        return preprocess(image)


def batched(iterable, batch_size):
    for start in range(0, len(iterable), batch_size):
        yield start, iterable[start:start + batch_size]


image_embeddings = []
crop_paths = metadata["crop_path"].tolist()

with torch.no_grad():
    for _, batch_paths in tqdm(list(batched(crop_paths, BATCH_SIZE)), desc="Encoding train+valid crop images"):
        images = torch.stack([load_preprocessed_crop(path) for path in batch_paths]).to(DEVICE)
        features = model.encode_image(images)
        features = features / features.norm(dim=-1, keepdim=True)
        image_embeddings.append(features.cpu().numpy())

embeddings = np.concatenate(image_embeddings, axis=0).astype("float32")
np.save(OUTPUTS_DIR / "embeddings.npy", embeddings)

embeddings_metadata = metadata.copy()
embeddings_metadata.to_csv(OUTPUTS_DIR / "embeddings_metadata.csv", index=False)

print(embeddings.shape)
print(f"Saved train+valid visual embeddings to {OUTPUTS_DIR / 'embeddings.npy'}")

Encoding train+valid crop images: 100%|██████████| 125/125 [03:07<00:00,  1.50s/it]

(7947, 512)
Saved train+valid visual embeddings to C:\Users\Juan Ramirez\Documents\CLIPmod\clip_module\outputs\embeddings.npy


In [8]:
assert embeddings.shape[0] == len(metadata), "Number of embeddings must match train+valid metadata rows"
assert set(embeddings_metadata["split"].unique()).issubset({"train", "valid"}), "Indexed embeddings must exclude test split"
assert not (embeddings_metadata["split"] == "test").any(), "Test crops must not be stored in embeddings.npy"
assert not np.isnan(embeddings).any(), "Embeddings contain NaN"
norms = np.linalg.norm(embeddings, axis=1)
assert np.allclose(norms, 1.0, atol=1e-4), f"Embeddings are not L2-normalized: min={norms.min()}, max={norms.max()}"
pd.Series(norms).describe()

count    7.947000e+03
mean     1.000000e+00
std      4.687663e-08
min      9.999998e-01
25%      9.999999e-01
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

In [9]:
prompt_rows = []
class_text_embeddings = []

with torch.no_grad():
    for class_id, class_name in CLASS_PROMPT_NAMES.items():
        prompts = [template.format(class_name) for template in PROMPT_TEMPLATES]
        tokens = tokenizer(prompts).to(DEVICE)
        prompt_features = model.encode_text(tokens)
        prompt_features = prompt_features / prompt_features.norm(dim=-1, keepdim=True)
        class_feature = prompt_features.mean(dim=0, keepdim=True)
        class_feature = class_feature / class_feature.norm(dim=-1, keepdim=True)
        class_text_embeddings.append(class_feature.cpu().numpy())

        for template, prompt in zip(PROMPT_TEMPLATES, prompts):
            prompt_rows.append({
                "class_id": class_id,
                "class_name": class_name,
                "prompt": prompt,
                "prompt_template": template,
            })

text_embeddings = np.concatenate(class_text_embeddings, axis=0).astype("float32")
text_prompts_metadata = pd.DataFrame(prompt_rows)
text_class_summary = pd.DataFrame({"class_id": list(CLASS_PROMPT_NAMES.keys()), "class_name": list(CLASS_PROMPT_NAMES.values())})

np.save(OUTPUTS_DIR / "text_embeddings.npy", text_embeddings)
text_prompts_metadata.to_csv(OUTPUTS_DIR / "text_prompts_metadata.csv", index=False)
text_class_summary.to_csv(OUTPUTS_DIR / "text_embeddings_metadata.csv", index=False)

assert not text_prompts_metadata["prompt"].str.contains(r"\b[0-4]\b", regex=True).any(), "Prompts must use class names, not ids"
assert np.allclose(np.linalg.norm(text_embeddings, axis=1), 1.0, atol=1e-4)

print(text_embeddings.shape)
text_prompts_metadata.head(10)

(5, 512)


,class_id,class_name,prompt,prompt_template
0,0,gun,a baggage x-ray image of a gun,a baggage x-ray image of a {}
1,0,gun,an x-ray crop containing a gun,an x-ray crop containing a {}
2,0,gun,a prohibited item: gun,a prohibited item: {}
3,0,gun,a security x-ray image showing a gun,a security x-ray image showing a {}
4,0,gun,a cropped x-ray baggage image of a gun,a cropped x-ray baggage image of a {}
5,1,knife,a baggage x-ray image of a knife,a baggage x-ray image of a {}
6,1,knife,an x-ray crop containing a knife,an x-ray crop containing a {}
7,1,knife,a prohibited item: knife,a prohibited item: {}
8,1,knife,a security x-ray image showing a knife,a security x-ray image showing a {}
9,1,knife,a cropped x-ray baggage image of a knife,a cropped x-ray baggage image of a {}
